# 🧬 Comparative TP53 Sequence Analysis

## Comparative In-Silico Analysis of TP53 Mutation Hotspots Between Humans and Elephants

**Research Project:** Elephant TP53 Hotspot Mapping  
**Researcher:** Ritika Rajendra Rawat  
**Degree:** MSc Bioinformatics  
**Institution:** University of Mumbai

---

## 🔬 Research Objective

This notebook implements a reproducible comparative sequence-analysis workflow for examining TP53-related protein sequences from humans and elephants.

The workflow includes:

- sequence quality control;
- reference sequence identification;
- protein sequence preprocessing;
- pairwise sequence comparison;
- sequence identity estimation;
- alignment-aware residue mapping;
- canonical human TP53 hotspot mapping;
- comparative sequence characterization;
- visualization;
- reproducible result generation.

### Canonical TP53 hotspot positions

- R175
- G245
- R248
- R249
- R273
- R282

> This notebook performs comparative bioinformatics analysis. It is not a clinical diagnostic system, cancer-risk predictor, or validated model of elephant cancer resistance.

In [ ]:
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from Bio import SeqIO
from Bio.Align import PairwiseAligner

print('Libraries loaded successfully.')

In [ ]:
def find_repository_root():
    current = Path.cwd().resolve()

    candidates = [
        current,
        current.parent,
        current.parent.parent,
        current.parent.parent.parent
    ]

    for path in candidates:
        if (path / 'data').exists() and (path / 'README.md').exists():
            return path

    raise FileNotFoundError('Could not locate repository root.')


ROOT = find_repository_root()

DATA_DIR = ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
    
RESULTS_DIR = ROOT / 'results'
FIGURES_DIR = ROOT / 'figures'

RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

INPUT_FASTA = PROCESSED_DIR / 'TP53_clean.fasta'

print('Repository:', ROOT)
print('Input:', INPUT_FASTA)

In [ ]:
if not INPUT_FASTA.exists():
    raise FileNotFoundError(
        f'Required input file was not found: {INPUT_FASTA}'
    )

records = list(SeqIO.parse(INPUT_FASTA, 'fasta'))

print(f'FASTA records loaded: {len(records)}')

for record in records:
    print(f'{record.id}: {len(record.seq)} aa')

In [ ]:
VALID_AMINO_ACIDS = set('ACDEFGHIKLMNPQRSTVWY')


def clean_sequence(sequence):
    return str(sequence).upper().replace(' ', '').replace('\n', '')


cleaned_records = []

for record in records:
    sequence = clean_sequence(record.seq)
    invalid = set(sequence) - VALID_AMINO_ACIDS

    if invalid:
        print(
            f'Excluded {record.id}: invalid residues = {sorted(invalid)}'
        )
        continue

    cleaned_records.append({
        'id': record.id,
        'description': record.description,
        'sequence': sequence
    })

print('Valid sequences retained:', len(cleaned_records))

In [ ]:
def identify_human_reference(records):
    preferred_terms = [
        'P04637',
        'P53_HUMAN',
        'TP53_HUMAN',
        'HUMAN_TP53',
        'Human_TP53'
    ]

    for term in preferred_terms:
        for record in records:
            searchable = (
                record['id'] + ' ' + record['description']
            ).upper()

            if term.upper() in searchable:
                return record

    return None


human_record = identify_human_reference(cleaned_records)

if human_record is None:
    raise ValueError('Human TP53 reference sequence could not be identified.')

human_id = human_record['id']
human_sequence = human_record['sequence']

print('Human reference:', human_id)
print('Sequence length:', len(human_sequence))

In [ ]:
EXPECTED_HUMAN_TP53_LENGTH = 393

if len(human_sequence) != EXPECTED_HUMAN_TP53_LENGTH:
    print(
        'WARNING:',
        f'detected length = {len(human_sequence)},',
        f'expected = {EXPECTED_HUMAN_TP53_LENGTH}'
    )
else:
    print('Human TP53 length validation: PASS')

# 🎯 Canonical TP53 Hotspots

The following human TP53 hotspot positions are used as reference coordinates:

| Position | Hotspot |
|---:|---|
| 175 | R175 |
| 245 | G245 |
| 248 | R248 |
| 249 | R249 |
| 273 | R273 |
| 282 | R282 |

Residue correspondence is determined through sequence alignment rather than assuming that identical numerical coordinates represent identical evolutionary positions.

In [ ]:
HOTSPOTS = {
    175: 'R175',
    245: 'G245',
    248: 'R248',
    249: 'R249',
    273: 'R273',
    282: 'R282'
}

print('Reference hotspot validation:\n')

for position, label in HOTSPOTS.items():
    residue = human_sequence[position - 1]
    print(f'{label}: reference residue = {residue}')

In [ ]:
aligner = PairwiseAligner()

aligner.mode = 'global'
aligner.match_score = 1.0
aligner.mismatch_score = 0.0
aligner.open_gap_score = -1.0
aligner.extend_gap_score = -0.1

print('Global pairwise alignment configured.')

In [ ]:
def calculate_identity(reference, query):
    alignment = aligner.align(reference, query)[0]

    aligned_reference = str(alignment[0])
    aligned_query = str(alignment[1])

    matches = 0
    comparable = 0

    for ref_residue, query_residue in zip(
        aligned_reference,
        aligned_query
    ):
        if ref_residue != '-' and query_residue != '-':
            comparable += 1

            if ref_residue == query_residue:
                matches += 1

    if comparable == 0:
        return np.nan

    return matches / comparable

In [ ]:
def map_reference_position(reference, query, reference_position):
    alignment = aligner.align(reference, query)[0]

    aligned_reference = str(alignment[0])
    aligned_query = str(alignment[1])

    reference_counter = 0

    for ref_residue, query_residue in zip(
        aligned_reference,
        aligned_query
    ):
        if ref_residue != '-':
            reference_counter += 1

        if reference_counter == reference_position:
            return query_residue

    return None

In [ ]:
identity_rows = []

for record in cleaned_records:
    identity = calculate_identity(
        human_sequence,
        record['sequence']
    )

    identity_rows.append({
        'id': record['id'],
        'length': len(record['sequence']),
        'identity_to_human': identity,
        'identity_to_human_percent': (
            identity * 100 if not np.isnan(identity) else np.nan
        )
    })

identity_df = pd.DataFrame(identity_rows)

identity_df.sort_values(
    'identity_to_human_percent',
    ascending=False
)

In [ ]:
hotspot_rows = []

for record in cleaned_records:
    row = {'id': record['id']}
    conserved_values = []

    for position, label in HOTSPOTS.items():
        human_residue = human_sequence[position - 1]

        query_residue = map_reference_position(
            human_sequence,
            record['sequence'],
            position
        )

        conserved = query_residue == human_residue

        row[f'{label}_human'] = human_residue
        row[f'{label}_query'] = (
            query_residue if query_residue is not None else '-'
        )
        row[f'{label}_conserved'] = conserved

        conserved_values.append(int(conserved))

    row['hotspot_conservation_percent'] = (
        np.mean(conserved_values) * 100
    )

    hotspot_rows.append(row)

hotspot_df = pd.DataFrame(hotspot_rows)

hotspot_df

In [ ]:
def amino_acid_features(sequence):
    length = len(sequence)
    result = {'length': length}

    for amino_acid in sorted(VALID_AMINO_ACIDS):
        result[f'fraction_{amino_acid}'] = (
            sequence.count(amino_acid) / length
        )

    return result


feature_rows = []

for record in cleaned_records:
    row = {'id': record['id']}
    row.update(amino_acid_features(record['sequence']))
    feature_rows.append(row)

feature_df = pd.DataFrame(feature_rows)

feature_df.head()

In [ ]:
comparative_df = (
    feature_df
    .merge(
        identity_df,
        on=['id', 'length'],
        how='left'
    )
    .merge(
        hotspot_df,
        on='id',
        how='left'
    )
)

comparative_df

In [ ]:
output_file = RESULTS_DIR / 'tp53_comparative_features.csv'

comparative_df.to_csv(
    output_file,
    index=False
)

print('Saved:', output_file)

In [ ]:
plot_df = comparative_df.sort_values(
    'identity_to_human_percent'
)

plt.figure(figsize=(10, 7))

plt.barh(
    plot_df['id'],
    plot_df['identity_to_human_percent']
)

plt.xlabel('Identity to human TP53 (%)')
plt.ylabel('Sequence')
plt.title('TP53 Sequence Identity Relative to Human Reference')

plt.tight_layout()

figure_path = FIGURES_DIR / 'tp53_identity_to_human.png'

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
heatmap_columns = [
    f'{label}_conserved'
    for label in HOTSPOTS.values()
]

heatmap_df = (
    hotspot_df[
        ['id'] + heatmap_columns
    ]
    .set_index('id')
    .astype(int)
)

heatmap_df.columns = list(HOTSPOTS.values())

plt.figure(figsize=(10, 7))

sns.heatmap(
    heatmap_df,
    annot=True,
    fmt='d',
    vmin=0,
    vmax=1,
    linewidths=0.5
)

plt.xlabel('Human TP53 hotspot')
plt.ylabel('Sequence')
plt.title('Alignment-Aware TP53 Hotspot Conservation')

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / 'tp53_hotspot_conservation.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

# 🔬 Results Interpretation

The workflow provides sequence-level measurements describing similarity to human TP53, alignment-aware correspondence of canonical TP53 hotspot positions, conservation or divergence at these positions, and amino-acid composition characteristics.

These findings represent comparative sequence observations. They do not establish elephant cancer resistance, functional equivalence, altered tumor-suppressor activity, clinical protection, or a causal mechanism of cancer susceptibility.

Additional evolutionary, structural, functional, and experimental evidence would be required for such conclusions.

# ⚠️ Limitations

- Sequence conservation does not necessarily imply functional equivalence.
- TP53-related sequences may represent canonical genes, duplicates, or retrogene-like copies.
- Sequence analysis does not measure protein expression or biochemical activity.
- Hotspot conservation alone cannot establish cancer susceptibility.
- Results depend on reference sequences and preprocessing.
- Pairwise alignment should be complemented by broader evolutionary analyses when making phylogenetic claims.